# ToS;DR LLM Evaluation Pipeline
This is where I will test out evaluating LLMs on ToS;DR dataset. After I get this right, I will merge this notebook with `100_tos_evaluation.ipynb` into one big evaluation pipeline.

## TODO: 
- incorporate the rubric into the evaluation dataframe.
- prepare the evaluation pipeline (its ok if it'lll have its own format for now)
- run the LLM evaluation pipeline.
- - i don't have a rubric for bad, good, or neutral. I'll need help from some LLM to do that.

In [24]:
# Trying to make tos_dr json files into dataframe.
import pandas as pd
import json

In [25]:
JSON_PATH = "../../generated_files/tos_dr"
SERVICES_JSON_PATH = f"{JSON_PATH}/tosdr_services_denormalized.json"
TOSDR_FILES_PATH = "../../generated_files/tos_dr"
TOPICS_SCORED_JSON_PATH = "../../generated_files/tosdr_topics_scored.json"

In [26]:
# Load the Amazon JSON file
with open(f"{JSON_PATH}/amazon_service.json", 'r') as f:
    data = json.load(f)

# Create a DataFrame from the JSON data
df_amazon = pd.json_normalize(
    data,
    record_path=['cases', 'points'],
    meta=[
        'service_name',
        ['cases', 'case_id'],
        ['cases', 'case_classification'],
        ['cases', 'case_title'],
        ['cases', 'case_description'],
        ['cases', 'case_topic'],        
    ]
)

df_amazon.head()

,point_id,point_title,point_source,point_analysis,point_quote_text,point_quote_start,point_quote_end,point_document_id,service_name,cases.case_id,cases.case_classification,cases.case_title,cases.case_description,cases.case_topic
0,1582,personal data is given to third parties,https://www.amazon.com/gp/help/customer/displa...,Amazon may release your data when they believe...,This includes exchanging information with othe...,8238.0,8361.0,38.0,Amazon,188,bad,This service gives your personal data to third...,Your personal data is or may be given to third...,Third Parties
1,5929,This service can share your personal informati...,https://www.amazon.com/gp/help/customer/displa...,Generated through the annotate view,The Amazon Group Companies are subject to the ...,4616.0,4952.0,1051.0,Amazon,188,bad,This service gives your personal data to third...,Your personal data is or may be given to third...,Third Parties
2,1122,The service uses your personal data for advert...,https://www.amazon.com/gp/help/customer/displa...,Amazon uses your personal data and your behavi...,"To serve you interest-based ads, we use inform...",525.0,643.0,39.0,Amazon,216,bad,Your personal data is used for advertising,Your interaction with the service and data you...,Advertising
3,5925,This service employs separate policies for dif...,https://www.amazon.com/gp/help/customer/displa...,Generated through the annotate view,"Please review our other policies, such as our ...",16304.0,16441.0,37.0,Amazon,200,neutral,Separate policies are employed for different p...,The user may need to read additional product-s...,Transparency
4,5923,This service forces users into binding arbitra...,https://www.amazon.com/gp/help/customer/displa...,Generated through the annotate view,Any dispute or claim relating in any way to yo...,13750.0,14060.0,37.0,Amazon,339,bad,You are forced into binding arbitration in cas...,This service forces users to use their own con...,Dispute Resolution


## Ingest all ToS;DR points

One row per point across every service in `tosdr_services_denormalized.json`. Services (or cases) without points are skipped.

In [27]:
POINT_META = [
    "service_name",
    ["cases", "case_id"],
    ["cases", "case_classification"],
    ["cases", "case_title"],
    ["cases", "case_description"],
    ["cases", "case_topic"],
]


def load_tosdr_points_df(json_path: str) -> pd.DataFrame:
    """Load all ToS;DR points from denormalized services JSON.

    Returns one row per point with the same columns as the single-service
    `pd.json_normalize(..., record_path=['cases', 'points'])` approach.
    Services without cases/points are skipped.
    """
    with open(json_path, "r") as f:
        services = json.load(f)

    services_with_points = []
    for service in services:
        cases_with_points = [
            case for case in service.get("cases", []) if case.get("points")
        ]
        if cases_with_points:
            services_with_points.append({**service, "cases": cases_with_points})

    if not services_with_points:
        return pd.DataFrame()

    return pd.json_normalize(
        services_with_points,
        record_path=["cases", "points"],
        meta=POINT_META,
    )

In [28]:
def format_topic_rubric(scores: list[dict]) -> str:
    """Format a topic's score descriptions into a multi-line rubric."""
    return "\n".join(
        f"{item['score']}: {item['description']}"
        for item in sorted(scores, key=lambda x: x["score"])
    )


def load_topic_rubrics(json_path: str) -> dict[str, str]:
    """Load topic title -> formatted score_rubric mapping."""
    with open(json_path, "r") as f:
        topics = json.load(f)
    return {
        topic["title"]: format_topic_rubric(topic["scores"])
        for topic in topics
    }


def attach_score_rubric(
    points_df: pd.DataFrame,
    topics_json_path: str,
    topic_col: str = "cases.case_topic",
) -> pd.DataFrame:
    """Add score_rubric column by matching each row's case topic."""
    rubrics = load_topic_rubrics(topics_json_path)
    out = points_df.copy()
    out["score_rubric"] = out[topic_col].map(rubrics)
    return out

In [ ]:
tos_points_df = attach_score_rubric(
    load_tosdr_points_df(SERVICES_JSON_PATH),
    TOPICS_SCORED_JSON_PATH,
)

,point_id,point_title,point_source,point_analysis,point_quote_text,point_quote_start,point_quote_end,point_document_id,service_name,cases.case_id,cases.case_classification,cases.case_title,cases.case_description,cases.case_topic,score_rubric
0,17470,The service provides information about how the...,https://telegram.org/privacy,Generated through the annotate view,<li>what we may use your personal data for;,3690.0,3734.0,2058.0,Telegram,227,good,Information is provided about how your persona...,The Privacy Policy explains the purposes for w...,Transparency,"-1: The terms and policies are inaccessible, u..."
1,17474,The service does not use third-party analytics...,https://telegram.org/privacy,Generated through the annotate view,We do not use cookies for profiling or adverti...,10113.0,10165.0,2058.0,Telegram,381,good,No third-party analytics or tracking platforms...,There are no Google Analytics or other trackin...,Third Parties,"-1: Personal data is shared with, or sold to, ..."
2,17480,You can delete your content from this service,https://telegram.org/privacy,Generated through the annotate view,"Deleting your account removes all messages, me...",22518.0,22723.0,2058.0,Telegram,175,good,You can delete your content from this service,You can ask the service to remove your content...,Right to Leave The Service,"-1: The user cannot freely terminate, or canno..."
3,8186,This service is only available to users of a c...,https://telegram.org/tos,None,Citizens of EU countries and the United Kingdo...,402.0,495.0,2059.0,Telegram,152,neutral,This service is only available to users over a...,The Services are intended for users who are at...,Governance,-1: The company retains wholly unilateral and ...
4,8188,This service does not sell your personal data,https://telegram.org/faq,None,"We don’t use your data for ad targeting, we do...",10819.0,10886.0,2060.0,Telegram,193,good,Your personal data is not sold,This service makes an explicit promise not to ...,Personal Data,"-1: Once collected, the user has no control ov..."


In [30]:
EXCLUDED_TOPICS = {"[Deprecated]", "Unclassified"}

tos_points_df = (
    tos_points_df[~tos_points_df["cases.case_topic"].isin(EXCLUDED_TOPICS)]
    .reset_index(drop=True)
)

print(f"Points after excluding deprecated/unclassified topics: {len(tos_points_df):,}")

Points after excluding deprecated/unclassified topics: 23,868


In [34]:
tos_points_df.head()

,point_id,point_title,point_source,point_analysis,point_quote_text,point_quote_start,point_quote_end,point_document_id,service_name,cases.case_id,cases.case_classification,cases.case_title,cases.case_description,cases.case_topic,score_rubric
0,17470,The service provides information about how the...,https://telegram.org/privacy,Generated through the annotate view,<li>what we may use your personal data for;,3690.0,3734.0,2058.0,Telegram,227,good,Information is provided about how your persona...,The Privacy Policy explains the purposes for w...,Transparency,"-1: The terms and policies are inaccessible, u..."
1,17474,The service does not use third-party analytics...,https://telegram.org/privacy,Generated through the annotate view,We do not use cookies for profiling or adverti...,10113.0,10165.0,2058.0,Telegram,381,good,No third-party analytics or tracking platforms...,There are no Google Analytics or other trackin...,Third Parties,"-1: Personal data is shared with, or sold to, ..."
2,17480,You can delete your content from this service,https://telegram.org/privacy,Generated through the annotate view,"Deleting your account removes all messages, me...",22518.0,22723.0,2058.0,Telegram,175,good,You can delete your content from this service,You can ask the service to remove your content...,Right to Leave The Service,"-1: The user cannot freely terminate, or canno..."
3,8186,This service is only available to users of a c...,https://telegram.org/tos,None,Citizens of EU countries and the United Kingdo...,402.0,495.0,2059.0,Telegram,152,neutral,This service is only available to users over a...,The Services are intended for users who are at...,Governance,-1: The company retains wholly unilateral and ...
4,8188,This service does not sell your personal data,https://telegram.org/faq,None,"We don’t use your data for ad targeting, we do...",10819.0,10886.0,2060.0,Telegram,193,good,Your personal data is not sold,This service makes an explicit promise not to ...,Personal Data,"-1: Once collected, the user has no control ov..."


In [31]:
tos_points_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23868 entries, 0 to 23867
Data columns (total 15 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   point_id                   23868 non-null  int64  
 1   point_title                23868 non-null  object 
 2   point_source               23753 non-null  object 
 3   point_analysis             23774 non-null  object 
 4   point_quote_text           22405 non-null  object 
 5   point_quote_start          22175 non-null  float64
 6   point_quote_end            22175 non-null  float64
 7   point_document_id          22398 non-null  float64
 8   service_name               23868 non-null  object 
 9   cases.case_id              23868 non-null  object 
 10  cases.case_classification  23868 non-null  object 
 11  cases.case_title           23868 non-null  object 
 12  cases.case_description     21631 non-null  object 
 13  cases.case_topic           23868 non-null  obj

In [32]:
print(f"Total points: {len(tos_points_df):,}")
print(f"Services: {tos_points_df['service_name'].nunique():,}")
print(f"Cases: {tos_points_df['cases.case_id'].nunique():,}")
print(f"\nClassifications:\n{tos_points_df['cases.case_classification'].value_counts()}")

Total points: 23,868
Services: 1,896
Cases: 243

Classifications:
cases.case_classification
neutral    10192
bad         6939
good        6248
blocker      489
Name: count, dtype: int64


In [33]:
# (Optional) Save dataframe to csv
tos_points_df.to_csv(f"{TOSDR_FILES_PATH}/tos_dr_points.csv", index=False)